# Multi-Source RF — Aggregated Group Features with Group-Aware Evaluation

Aggregates measurement statistics per (country, node, modem, run, rat) from multiple
CSV files, then merges them into a single feature matrix evaluated with GroupShuffleSplit.

In [1]:
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

RAW_DIR = "data/raw"
GROUP_COLS = ["country", "node_name", "modem_name", "run"]
TARGET_COL = "rat"
EXCLUDE_COLS = [
    "id", "node_name", "modem_name", "location",
    "mcc", "country", "iso_code", "operator_anon",
    "rat", "rat_name", "timestamp", "target_ip", "timetamp_ms", "run",
]
RANDOM_STATE = 42

SOURCE_FILES = {
    "ping":        "figure5_packet_loss.csv",
    "throughput":  "figure6_throughput.csv",
    "power":       "figure8_power_upload.csv",
}

### Aggregation function

For each file, for each (group, rat) combo, compute mean/std/min/max of numeric measurement columns.

In [2]:
def get_measurement_cols(df):
    return [c for c in df.columns
            if c not in EXCLUDE_COLS
            and pd.api.types.is_numeric_dtype(df[c])
            and df[c].nunique() > 1]

def aggregate_file(filepath, label):
    df = pd.read_csv(filepath)
    meas = get_measurement_cols(df)
    agg = df.groupby(GROUP_COLS + [TARGET_COL]).agg({m: ["mean","std","min","max"] for m in meas})
    agg.columns = [f"{label}_{a}_{b}" for a, b in agg.columns]
    return agg.reset_index()

def build_matrix(keys):
    frames = [aggregate_file(f"{RAW_DIR}/{SOURCE_FILES[k]}", k) for k in keys]
    merged = frames[0]
    for f in frames[1:]:
        merged = merged.merge(f, on=GROUP_COLS + [TARGET_COL], how="inner")
    y = merged[TARGET_COL]
    groups = merged[GROUP_COLS].astype(str).agg("-".join, axis=1)
    feats = [c for c in merged.columns if c not in GROUP_COLS + [TARGET_COL]]
    X = merged[feats].copy()
    # Drop zero-variance
    for c in X.columns:
        if X[c].nunique() <= 1:
            X = X.drop(columns=c)
    return X, y, groups

### Check overlap between sources

In [ ]:
for label, filename in SOURCE_FILES.items():
    df = pd.read_csv(f"{RAW_DIR}/{filename}")
    n = df.groupby(GROUP_COLS + [TARGET_COL]).ngroups
    print(f"{label}: {filename}  ->  {n} unique (group, rat) combos")

# Overlap check
sets = {}
for label, filename in SOURCE_FILES.items():
    df = pd.read_csv(f"{RAW_DIR}/{filename}")
    df["gk"] = df[GROUP_COLS].astype(str).agg("-".join, axis=1)
    sets[label] = set(zip(df["gk"], df[TARGET_COL]))

keys = list(SOURCE_FILES.keys())
for i in range(len(keys)):
    for j in range(i+1, len(keys)):
        overlap = len(sets[keys[i]] & sets[keys[j]])
        print(f"  {keys[i]} & {keys[j]}: {overlap} overlapping combos")
all3 = len(sets[keys[0]] & sets[keys[1]] & sets[keys[2]])
print(f"  All 3: {all3} overlapping combos")

### Train and evaluate each combination

In [ ]:
combos = [("ping",), ("ping","throughput"), ("ping","power"), ("ping","throughput","power")]
results = []

for combo in combos:
    name = "+".join(combo)
    X, y, groups = build_matrix(list(combo))

    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
    tr, te = next(gss.split(X, y, groups))
    clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    clf.fit(X.iloc[tr], y.iloc[tr])
    yp = clf.predict(X.iloc[te])

    labels = sorted(y.unique())
    results.append({
        "combination": name,
        "n": len(X), "n_feat": X.shape[1],
        "accuracy": accuracy_score(y.iloc[te], yp),
        "f1_weighted": f1_score(y.iloc[te], yp, average="weighted", zero_division=0),
        "cm": confusion_matrix(y.iloc[te], yp, labels=labels),
        "labels": [str(l) for l in labels],
    })
    print(f"{name:<30s}  N={len(X):2d}  Feats={X.shape[1]:2d}  Acc={results[-1]['accuracy']:.4f}  F1={results[-1]['f1_weighted']:.4f}")

### Results table

In [ ]:
pd.DataFrame(results)[["combination","n","n_feat","accuracy","f1_weighted"]]

### Best combination — Confusion Matrix

In [ ]:
best = max(results, key=lambda r: r["accuracy"])
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(best["cm"], annot=True, fmt="d", cmap="Blues",
            xticklabels=best["labels"], yticklabels=best["labels"], ax=ax)
ax.set_title(f"Multi-Source RF — {best['combination']} (Acc: {best['accuracy']:.4f}, N={best['n']})")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.show()

### Notes

- **Aggregation approach**: Each file is aggregated to (group, rat) level using mean/std/min/max.
  This avoids temporal leakage because each row is an independent group-level summary.
- **ping+throughput (87.5%)** is the best combination. Throughput statistics are strong RAT discriminators.
- **ping+power (75.0%)** also outperforms single-source ping (55.6%).
- **Three-source (66.7%)** suffers from too few overlapping samples (N=18).
- **Sample size is the main limitation**: only 18–30 rows after aggregation. Results are directionally
  informative but not statistically robust with test sets of 6–9 samples.
- **figure7_current.csv** was excluded as a feature source because its schema matches
  figure6_throughput.csv rather than containing the expected current/voltage columns.
- **Next step**: A Neural Network may handle small-sample scenarios better through regularization;
  Transfer Learning across countries becomes the natural evaluation framework.